# QDSV Bridge Colab 04 - IBM/Qiskit Artifact Demo

This notebook shows the public Bridge path for Qiskit-oriented users:

```text
problem intent -> controlled semantic spec -> Bridge artifact -> QASM/Qiskit inspection -> local simulation -> Bridge Report
```

Bridge does not replace Qiskit. It prepares a problem-first artifact that a Qiskit user can inspect, modify, simulate, or route toward IBM Quantum workflows when appropriate.

## 1. Install the public SDK and Qiskit tools

The notebook installs `qdsv-bridge` from PyPI. Qiskit packages are used only after Bridge returns an artifact, so the generated circuit remains inspectable by the user.

In [ ]:
!pip -q install -U "qdsv-bridge[qiskit]"

## 2. Connect to the public Bridge API

By default this notebook uses the public demo endpoint. If you are using a private deployment, set `QDSV_BRIDGE_API_URL` and optionally `QDSV_BRIDGE_API_KEY` before creating the client.

In [ ]:
import os
from IPython.display import Markdown, display

from qdsv_bridge import QDSVBridgeClient

API_URL = os.getenv("QDSV_BRIDGE_API_URL") or None
API_KEY = os.getenv("QDSV_BRIDGE_API_KEY") or None

client = QDSVBridgeClient(api_url=API_URL, api_key=API_KEY, timeout=60)
families = client.families()

print("API status:", families.get("status"))
print("Supported families:", ", ".join(sorted(families.get("families", {}).keys())))
print("API key required for:", families.get("public_limits", {}).get("api_key_required_for"))

## 3. Declare the problem before the circuit

This example uses the controlled fallback family `bounded_semantic_marking`. The point is to declare the state-space role, the marking goal, the requested artifact target, and the resource limits before asking for a Qiskit-oriented artifact.

In [ ]:
spec = {
    "family": "bounded_semantic_marking",
    "bridge_mode": "build",
    "state_space": {
        "kind": "finite_candidates",
        "candidate_count": 8,
        "candidate_id": "candidate",
    },
    "signals": ["eligibility_score", "risk_score"],
    "goal": {
        "kind": "marking",
        "predicate": "eligible_candidate",
    },
    "target": {
        "format": "qasm3",
        "backend_family": "qiskit",
    },
    "limits": {
        "max_qubits": 5,
        "max_depth": 160,
    },
}

spec

## 4. Build a Qiskit-oriented artifact with Bridge

The response includes the generated artifact, semantic preservation metadata, warnings, and digests. These are the trust and reproducibility handles that Bridge adds before the circuit enters the usual Qiskit workflow.

In [ ]:
artifact_package = client.build(spec)
artifact = artifact_package["artifact"]
qasm3_source = artifact.get("content", "")

print("status:", artifact_package.get("status"))
print("family:", artifact_package.get("family"))
print("mode:", artifact_package.get("bridge_mode"))
print("artifact format:", artifact.get("format"))
print("circuit origin:", artifact_package.get("circuit_origin"))
print("semantic preservation:", artifact_package.get("semantic_preservation"))
print("warnings:")
for warning in artifact_package.get("warnings", []):
    print("-", warning)

print("\nDigests:")
for key, value in artifact_package.get("digests", {}).items():
    print(f"- {key}: {value}")

print("\nQASM3 artifact:\n")
print(qasm3_source)

## 5. Load the artifact into Qiskit

This step is intentionally explicit. Qiskit users keep control: they can inspect the generated artifact, edit it, or replace the semantic oracle insertion point with a custom construction.

In [ ]:
from qiskit import qasm3

circuit = None
try:
    circuit = qasm3.loads(qasm3_source)
    print(circuit)
except Exception as exc:
    print("Qiskit could not import this QASM3 artifact in the current environment.")
    print(type(exc).__name__ + ":", exc)
    print("\nThe artifact is still saved and inspectable as OpenQASM 3 text.")

with open("bridge_ibm_qiskit_artifact.qasm", "w", encoding="utf-8") as f:
    f.write(qasm3_source)

print("Saved: bridge_ibm_qiskit_artifact.qasm")

## 6. Simulate locally with Qiskit Aer

If the QASM3 artifact loaded successfully, run the circuit locally. This keeps the demo free and reproducible before any IBM Quantum hardware/runtime path is considered.

In [ ]:
if circuit is None:
    print("Skipping simulation because no Qiskit circuit was loaded.")
else:
    from qiskit import transpile
    from qiskit_aer import AerSimulator

    simulator = AerSimulator()
    compiled = transpile(circuit, simulator)
    job = simulator.run(compiled, shots=1024)
    result = job.result()
    counts = result.get_counts()
    print("Counts:")
    print(counts)

## 7. Optional IBM Quantum Runtime path

This notebook does not require IBM credentials. If you already use IBM Quantum, the loaded Qiskit circuit can be routed through your normal Qiskit Runtime workflow after inspection and any edits you need.

Example outline only:

```python
# !pip install -U "qiskit-ibm-runtime>=0.40,<0.41"
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# service = QiskitRuntimeService(channel="ibm_quantum", token="YOUR_IBM_QUANTUM_TOKEN")
# backend = service.least_busy(operational=True, simulator=False)
# sampler = Sampler(backend)
# job = sampler.run([circuit])
# print(job.result())
```

Bridge's role is upstream of that execution step: preserve problem intent, generate an auditable artifact, and attach reproducibility metadata.

## 8. Generate the Bridge Report

The report is the shareable evidence document. It records what was requested, what artifact was generated, what was preserved, and what warnings or limits should be reviewed before execution.

In [ ]:
report = client.report(spec, mode="build", format="markdown")
display(Markdown(report["content"]))

with open("bridge_report_ibm_qiskit_demo.md", "w", encoding="utf-8") as f:
    f.write(report["content"])

print("Saved: bridge_report_ibm_qiskit_demo.md")